In [2]:
"""
AES(Advanced Encryption Standard) 블록암호 실습

- 블록 크기: 128비트(16바이트) 고정
- 키 길이: 128 / 192 / 256비트 (16 / 24 / 32바이트) -> AES-128, AES-192, AES-256
- 대칭키 암호, DES 대체로 널리 사용 (웹·디스크·VPN 등)
- 운용 모드: ECB(교육용), CBC, GCM(기밀성+무결성, nonce 필요) 예제

필요 패키지: pip install pycryptodome
"""

from __future__ import annotations

try:
    from Crypto.Cipher import AES
    from Crypto.Random import get_random_bytes
    from Crypto.Util.Padding import pad, unpad
except ImportError as e:  # pragma: no cover
    raise ImportError(
        "pycryptodome이 필요합니다. 터미널에서: python -m pip install pycryptodome"
    ) from e


def normalize_aes_key(key: str | bytes, key_bits: int = 256) -> bytes:
    """
    AES 키를 key_bits에 맞는 길이로 맞춤 (부족 시 0 패딩, 초과 시 잘라냄).

    교육용 단순화입니다. 실무에서는 비밀번호에서 키를 뽑을 때 PBKDF2, scrypt 등 KDF를 씁니다.
    """
    if key_bits not in (128, 192, 256):
        raise ValueError("key_bits는 128, 192, 256 중 하나여야 합니다.")
    n = key_bits // 8
    if isinstance(key, str):
        key = key.encode("utf-8")
    if len(key) < n:
        return key + b"\x00" * (n - len(key))
    if len(key) > n:
        return key[:n]
    return key


def encrypt_ecb(plaintext: str, key: str | bytes, *, key_bits: int = 256) -> bytes:
    """ECB: 같은 평문 블록 -> 같은 암호문 (패턴 노출). 이해용 데모 위주."""
    k = normalize_aes_key(key, key_bits)
    cipher = AES.new(k, AES.MODE_ECB)
    data = pad(plaintext.encode("utf-8"), AES.block_size)
    return cipher.encrypt(data)


def decrypt_ecb(ciphertext: bytes, key: str | bytes, *, key_bits: int = 256) -> str:
    k = normalize_aes_key(key, key_bits)
    cipher = AES.new(k, AES.MODE_ECB)
    data = unpad(cipher.decrypt(ciphertext), AES.block_size)
    return data.decode("utf-8")


def encrypt_cbc(
    plaintext: str, key: str | bytes, iv: bytes, *, key_bits: int = 256
) -> bytes:
    """CBC: IV는 블록 크기(16바이트)와 같아야 함. 메시지마다 새 IV 권장."""
    if len(iv) != AES.block_size:
        raise ValueError(f"IV 길이는 {AES.block_size}바이트여야 합니다.")
    k = normalize_aes_key(key, key_bits)
    cipher = AES.new(k, AES.MODE_CBC, iv=iv)
    data = pad(plaintext.encode("utf-8"), AES.block_size)
    return cipher.encrypt(data)


def decrypt_cbc(
    ciphertext: bytes, key: str | bytes, iv: bytes, *, key_bits: int = 256
) -> str:
    if len(iv) != AES.block_size:
        raise ValueError(f"IV 길이는 {AES.block_size}바이트여야 합니다.")
    k = normalize_aes_key(key, key_bits)
    cipher = AES.new(k, AES.MODE_CBC, iv=iv)
    data = unpad(cipher.decrypt(ciphertext), AES.block_size)
    return data.decode("utf-8")


def encrypt_gcm(
    plaintext: str,
    key: str | bytes,
    *,
    key_bits: int = 256,
    nonce: bytes | None = None,
) -> tuple[bytes, bytes, bytes]:
    """
    AES-GCM: 암호문 + 인증 태그 + nonce를 반환.
    nonce는 복호화 시 동일해야 함(통신 시 암호문과 함께 저장/전송, 비밀 아님).
    기본 12바이트 nonce(권장 길이).
    """
    k = normalize_aes_key(key, key_bits)
    if nonce is None:
        nonce = get_random_bytes(12)
    elif len(nonce) not in (8, 12, 13, 14, 15, 16):
        raise ValueError("GCM nonce 길이는 8~16바이트(관례적으로 12)를 사용하세요.")
    cipher = AES.new(k, AES.MODE_GCM, nonce=nonce)
    data = plaintext.encode("utf-8")
    ciphertext, tag = cipher.encrypt_and_digest(data)
    return ciphertext, tag, nonce


def decrypt_gcm(
    ciphertext: bytes,
    tag: bytes,
    nonce: bytes,
    key: str | bytes,
    *,
    key_bits: int = 256,
) -> str:
    k = normalize_aes_key(key, key_bits)
    cipher = AES.new(k, AES.MODE_GCM, nonce=nonce)
    plain = cipher.decrypt_and_verify(ciphertext, tag)
    return plain.decode("utf-8")


def demo() -> None:
    secret = "Sixteen byte key!!"  # 16바이트 UTF-8 -> AES-128
    message = "Hello AES - wide block cipher (256-bit key demo below)."

    print("=== AES lab (PyCryptodome) ===\n")

    # AES-256 / ECB
    key256 = "0123456789abcdef0123456789abcdef"  # 32 ASCII chars = 256-bit
    ct_ecb = encrypt_ecb(message, key256, key_bits=256)
    print("[AES-256 ECB] ciphertext (hex):", ct_ecb.hex())
    print("[AES-256 ECB] plaintext:", decrypt_ecb(ct_ecb, key256, key_bits=256))

    # CBC (fixed IV for demo only)
    iv = b"\x02" * AES.block_size
    ct_cbc = encrypt_cbc(message, key256, iv, key_bits=256)
    print("\n[AES-256 CBC] IV (hex):", iv.hex())
    print("[AES-256 CBC] ciphertext (hex):", ct_cbc.hex())
    print("[AES-256 CBC] plaintext:", decrypt_cbc(ct_cbc, key256, iv, key_bits=256))

    # GCM (random nonce each run)
    ct_gcm, tag_gcm, nonce_gcm = encrypt_gcm(message, key256, key_bits=256)
    print("\n[AES-256 GCM] nonce (hex):", nonce_gcm.hex())
    print("[AES-256 GCM] ciphertext (hex):", ct_gcm.hex())
    print("[AES-256 GCM] tag (hex):", tag_gcm.hex())
    print(
        "[AES-256 GCM] plaintext:",
        decrypt_gcm(ct_gcm, tag_gcm, nonce_gcm, key256, key_bits=256),
    )

    # ECB block repetition (AES block = 16 bytes)
    block = "A" * 16
    ct_rep = encrypt_ecb(block * 2, key256, key_bits=256)
    b0, b1 = ct_rep[:16], ct_rep[16:32]
    print("\n[ECB limit] two identical 16-byte blocks -> same cipher blocks?", b0 == b1)

    # AES-128 with 16-byte passphrase
    ct128 = encrypt_ecb("Short msg AES-128.", secret, key_bits=128)
    print("\n[AES-128 ECB] decrypt:", decrypt_ecb(ct128, secret, key_bits=128))


if __name__ == "__main__":
    demo()


=== AES lab (PyCryptodome) ===

[AES-256 ECB] ciphertext (hex): a8c5c9aa2d17b1b9798f204504b301b764d5b843f63294a1108cceba561db8786cd8a5084de0360d97005ae38d105e4a8fac3baa1ebd1144210a04867b943f73
[AES-256 ECB] plaintext: Hello AES - wide block cipher (256-bit key demo below).

[AES-256 CBC] IV (hex): 02020202020202020202020202020202
[AES-256 CBC] ciphertext (hex): 9a244075def4f299f751d55d51bd862f223b71ce8ac9901ab4e8ef786d8a42a137e5caff6ddb200969303f85bdbb54e27ad905c348356d3544f5aab2c882e3f5
[AES-256 CBC] plaintext: Hello AES - wide block cipher (256-bit key demo below).

[AES-256 GCM] nonce (hex): a44baeefdb14394ed019c063
[AES-256 GCM] ciphertext (hex): 6446feb603a43228dac21f117a8f284863c3a6ff517848144c5e7b9721d4f3f4a49ab3fc3acdd5523ad2b35cdd3c40da4fe2d340ea96ce
[AES-256 GCM] tag (hex): 9f635b6eaa5c838d89c4bb60e6099973
[AES-256 GCM] plaintext: Hello AES - wide block cipher (256-bit key demo below).

[ECB limit] two identical 16-byte blocks -> same cipher blocks? True

[AES-128 ECB] decryp